In [12]:
import numpy as np

from skbio.stats.composition import clr, clr_inv
from scipy.spatial.distance import euclidean

import plotly.graph_objects as go

def aitchison(x, y):
    return euclidean(clr(x), clr(y))

In [14]:
# Point for both sets of rings
point = np.array([0.7, 0.2, 0.1])

# --- Function for the standard deviation rings ---
def create_ring(center, radius, n_points=100):
    """Generates ternary coordinates for a circle around a point."""
    t = np.linspace(0, 2*np.pi, n_points)
    x_ring = center[0] + radius * np.cos(t)
    y_ring = center[1] + radius * np.sin(t)
    z_ring = 1 - x_ring - y_ring
    return x_ring, y_ring, z_ring

# --- Functions for CLR Circles ---

def get_clr_ring(center_clr, radius, n_points=100):
    t = np.linspace(0, 2*np.pi, n_points)
    
    v1 = np.array([1, -1, 0]) / np.sqrt(2)
    v2 = np.array([1, 1, -2]) / np.sqrt(6)
    
    ring_points_clr = []
    for angle in t:
        p_clr = center_clr + radius * (np.cos(angle) * v1 + np.sin(angle) * v2)
        ring_points_clr.append(clr_inv(p_clr))
    
    return np.array(ring_points_clr)

# --- Plotting ---
fig = go.Figure()

# Add the Rings (Standard Deviations)
radii = [0.03, 0.06, 0.09]  # 1SD, 2SD, 3SD equivalents
for i, r in enumerate(radii):
    r_a, r_b, r_c = create_ring(point, r)
    fig.add_trace(go.Scatterternary(
        a=r_a, b=r_b, c=r_c,
        mode='lines',
        line=dict(color='green', width=1.5, dash='dot'), # Dotted green for pixel-perfect
        name=f'Circle' if i == 0 else '', # Label only the first one
        showlegend=True if i == 0 else False
    ))


# Add CLR-Transformed Rings (Red)
point_clr = clr(point)
# clr_radii = [0.4, 0.8, 1.2] # Adjust as needed
clr_radii = [0.2, 0.4, 0.6] 
for i, r in enumerate(clr_radii):
    ring_simplex = get_clr_ring(point_clr, r)
    fig.add_trace(go.Scatterternary(
        a=ring_simplex[:, 0], b=ring_simplex[:, 1], c=ring_simplex[:, 2],
        mode='lines',
        line=dict(color='red', width=1.5, dash='solid'), # Solid red for CLR
        name=f'CLR Circle' if i == 0 else '', # Label only the first one
        showlegend=True if i == 0 else False
    ))

# Add the Cross (The Data Point)
fig.add_trace(go.Scatterternary(
    a=[point[0]], b=[point[1]], c=[point[2]],
    mode='markers',
    marker=dict(symbol='x', size=12, color='black'),
    name='Mean Value'
))

# Layout and Titles
fig.update_layout(
    ternary=dict(
        aaxis_title='Component A',
        baxis_title='Component B',
        caxis_title='Component C',
        sum=1 # Ensure it sums to 1
    ),
    title="Comparison of Standard (Green) and CLR-Transformed (Red) Rings",
    showlegend=True,
    width=750, height=650 
)

fig.show()